# Day 2 Laboratory Exercise: Turning A Sensor Feed Into A Dataset

The walkthrough cleaned a catalogue whose defects were visible from `info` and
`describe`. This laboratory exercise uses a genuinely raw feed, and none of its
defects announce themselves that way.

Eight low-cost particulate matter sensors reported from the Skopje area
throughout January 2024. Each one wrote a file of its own, roughly every two
and a half minutes, giving 137223 readings in total. `pm10` and `pm25` are
concentrations of airborne particles in micrograms per cubic metre. The
European Union sets a daily limit of 50 micrograms per cubic metre for PM10,
and your job by the end of the laboratory exercise is to say how many days in
January exceeded it.

That question has a single right answer, and the naive route to it produces the
wrong one.

## How To Work Through This

There are eight tasks. The first seven each state a goal, give you a cell
marked `# YOUR CODE HERE` that names the variables the check expects, and
follow it with a check cell that either confirms your answer or says what is
wrong. Task 8 is written for you and saves the result.

Task 3 asks you to notice something and then explicitly tells you to set it
aside. Do set it aside. Tasks 4 and 5 are about other things. The point of the
laboratory exercise arrives in Task 7 and it depends on your having moved on in
between.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path.cwd().parent / "data"
SENSOR_DIR = DATA_DIR / "sensors"
RANDOM_STATE = 42

# The European daily limit for PM10, in micrograms per cubic metre.
PM10_DAILY_LIMIT = 50

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 110)


def check(condition, success, failure):
    """Report whether a task was completed correctly."""
    print(success if condition else failure)


print("Sensor files found:", len(list(SENSOR_DIR.glob("station_*.csv"))))

## Task 1: Combine Eight Files Into One Table

`DATA_DIR / "sensors"` holds one CSV per station, named `station_<id>.csv`.
Each has three columns, which are `timestamp`, `pm10`, and `pm25`. The station
identifier appears only in the file name, so it has to be recovered from there
and added as a column, otherwise the readings become anonymous the moment the
files are concatenated.

Build a single DataFrame called `readings` with four columns, being
`timestamp`, `pm10`, `pm25`, and `station_id`. Leave `timestamp` as text for
now, because Task 2 is about parsing it.

In [ ]:
# YOUR CODE HERE
# Read every file in SENSOR_DIR, add a `station_id` column taken from the file
# name, and concatenate them into a DataFrame called `readings`.

In [ ]:
check(
    "readings" in dir()
    and len(readings) == 137223
    and readings["station_id"].nunique() == 8
    and {"timestamp", "pm10", "pm25", "station_id"} <= set(readings.columns),
    "Task 1 complete. 137223 readings from 8 stations.",
    "Not right yet. Expected a DataFrame called `readings` with 137223 rows, a "
    "`station_id` column, and 8 distinct stations.",
)

Two and a half minutes between readings for a month gives roughly 17000 rows
per station, and the totals differ because the stations were not all online for
the same amount of time. Task 3 quantifies that.

## Task 2: Parse The Timestamps And Notice What Goes Wrong

Every station wrote its own file with its own export settings, and they did not
agree. Look at the first timestamp of two different stations before you parse
anything.

```python
print(pd.read_csv(SENSOR_DIR / "station_20701.csv")["timestamp"].iloc[0])
print(pd.read_csv(SENSOR_DIR / "station_60237.csv")["timestamp"].iloc[0])
```

Six files record the instant in UTC with no offset attached. Two record it in
Skopje local time and say so with a trailing `+01:00`. Both are honest
descriptions of the same kind of moment, and mixing them without saying which
is which shifts two stations by an hour against the other six.

Parse the column into a genuine timezone-aware datetime in UTC, and overwrite
`readings["timestamp"]` with it. Try it first without `utc=True`, inside a
`try`, and read the error it raises, because the message names the fix and is
worth seeing once.

In [ ]:
# YOUR CODE HERE
# First parse without utc=True inside a try and print the error it raises.
# Then parse properly and assign the result back to readings["timestamp"].

In [ ]:
check(
    "readings" in dir()
    and isinstance(readings["timestamp"].dtype, pd.DatetimeTZDtype)
    and str(readings["timestamp"].dtype.tz) == "UTC"
    and str(readings["timestamp"].min()) == "2024-01-01 00:00:11+00:00",
    "Task 2 complete. Every reading is now a UTC instant and the month runs "
    "from 2024-01-01 to 2024-01-31.",
    "Not right yet. Expected readings['timestamp'] to be a UTC datetime "
    "column. Pass utc=True to pd.to_datetime.",
)

Two timestamps written in different conventions cannot share a column until
something decides which convention wins, and `utc=True` is how you decide.
pandas refuses the ambiguous parse and names the fix in the message, which is
the best case. Older versions answered quietly instead, returning a column of
loose Python objects that printed like dates and broke a much later cell, so a
notebook written a few years ago will tell you to inspect the dtype rather than
to read an error.

## Task 3: Profile Every Station Separately

Never summarise a multi-station feed before you have looked at the stations one
at a time. An average across sensors hides a broken sensor completely.

Build a DataFrame called `profile`, indexed by `station_id`, with four columns:

- `readings`, the number of rows,
- `days`, the number of distinct calendar days the station reported on,
- `median_pm10`, the median PM10 reading, and
- `max_pm10`, the largest PM10 reading.

`groupby("station_id").agg(...)` does all four. For `days`, count distinct
values of `timestamp.dt.normalize()`.

In [ ]:
# YOUR CODE HERE
# Build a DataFrame called `profile`, indexed by station_id, with columns
# readings, days, median_pm10, and max_pm10.

In [ ]:
check(
    "profile" in dir()
    and len(profile) == 8
    and {"readings", "days", "median_pm10", "max_pm10"} <= set(profile.columns)
    and int(profile["median_pm10"].idxmin()) == 78456,
    "Task 3 complete. Read the median_pm10 column before you move on.",
    "Not right yet. Expected a DataFrame called `profile` indexed by station_id "
    "with columns readings, days, median_pm10, and max_pm10.",
)

### Read That Table Before Continuing

Three things in it are worth naming now.

Station 78188 reported on 24 days rather than 31, and station 81853 on 30. Two
stations therefore have holes of several days, which will matter once anything
is averaged across stations.

Station 60237 has a maximum PM10 of exactly 1999.90, which is a suspiciously
round number for a physical measurement. Task 4 is about that.

Station 78456 has a median PM10 of 1.08, while the other seven sit between 8.50
and 60.63. It is lower than every one of them by a factor of at least seven, in
the same city in the same month.

Note that last figure down, and then set it aside. The next two tasks are about
other things.

## Task 4: Find The Readings That Are Not Measurements

A physical instrument has a range, and a reading at the very top of that range
usually means the true value was somewhere above it rather than exactly on it.
Such a reading is not a measurement, it is the sensor saying it ran out of
scale.

Look at every PM10 reading of 1900 or more, with `value_counts`. Genuine
readings are spread across many distinct values. A ceiling reveals itself as
one value repeated.

Count how many readings sit at that ceiling, store the count in
`n_saturated`, then replace those readings with `NaN` in `readings["pm10"]`.
Replacing rather than dropping keeps the row, because its `pm25` value and its
timestamp are still perfectly good.

In [ ]:
# YOUR CODE HERE
# Inspect readings of 1900 or more, count the ones at the ceiling into
# `n_saturated`, and set them to NaN in readings["pm10"].

In [ ]:
check(
    "n_saturated" in dir()
    and n_saturated == 15
    and readings["pm10"].max() < 1999.9
    and int(readings["pm10"].isna().sum()) == 15,
    "Task 4 complete. 15 readings sat at the 1999.90 ceiling, all of them at "
    "station 60237.",
    "Not right yet. Expected 15 readings at 1999.90 and those 15 replaced with "
    "NaN, leaving 15 missing PM10 values.",
)

Among the readings of 1900 or more, four are distinct values occurring once
each, and fifteen are the identical value 1999.90. Fifteen independent
measurements landing on the same two decimal places does not happen, and no
reading anywhere in the dataset is higher. That is the top of the instrument's
scale, and every one of those fifteen moments was dirtier than the number says.

Note which direction the error runs. Discarding them biases the month downward,
and keeping them also biases it downward, because the true values were higher.
There is no choice here that is free of consequence, which is why it belongs in
the write-up rather than only in the code.

## Task 5: Check That The Stations Are Where They Claim To Be

`DATA_DIR / "stations.csv"` holds the metadata, which is one row per station
with a latitude and a longitude. Metadata is data, and it gets the same
treatment as everything else.

Skopje sits near latitude 42.0 and longitude 21.4. Read the file, find any
station whose coordinates are not plausibly in the city, store its identifier
in `mislocated_id`, and then remove that station's readings from `readings`.

In [ ]:
# YOUR CODE HERE
# Read stations.csv, find the station that is not in Skopje, store its id in
# `mislocated_id`, and drop its readings from `readings`.

In [ ]:
check(
    "mislocated_id" in dir()
    and mislocated_id == 78844
    and readings["station_id"].nunique() == 7
    and len(readings) == 119526,
    "Task 5 complete. Station 78844 reports coordinates in Belgium, and 119526 "
    "readings from 7 stations remain.",
    "Not right yet. Expected mislocated_id to be 78844 and readings to be left "
    "with 119526 rows from 7 stations.",
)

Station 78844 gives its position as latitude 50.854 and longitude 2.860, which
is roughly 1600 kilometres away in Belgium. Its readings are real, they are
simply not readings of Skopje, and the network catalogued it as a Skopje
station anyway.

That is a defect a check can catch, which is exactly why it is worth writing
the check. A rule that says every station must lie within a sensible distance
of the city would have rejected this row automatically. Hold on to that
thought, because Task 7 turns on a defect that no rule of that kind would have
caught.

## Task 6: Build The City Series And Answer The Question

Now produce the number the laboratory exercise was set to produce.

Readings arrive at irregular moments, so they have to be put on a regular grid
before stations can be compared. Resample each station to an hourly mean, then
average across stations to get one city-wide series.

```python
hourly = (
    readings.set_index("timestamp")
    .groupby("station_id")["pm10"]
    .resample("1h")
    .mean()
    .unstack(0)
)
```

That gives a table of 744 hours by 7 stations. From it, build:

- `city_hourly`, the mean across stations for each hour,
- `city_daily`, the daily mean of `city_hourly`, and
- `days_over_limit`, the number of days whose mean exceeds `PM10_DAILY_LIMIT`.

Plot `city_daily` with the limit drawn on it.

In [ ]:
# YOUR CODE HERE
# Build `hourly`, then `city_hourly`, `city_daily`, and `days_over_limit`.
# Plot the daily series with the limit marked.

In [ ]:
check(
    "days_over_limit" in dir()
    and days_over_limit == 14
    and "city_hourly" in dir()
    and abs(city_hourly.mean() - 60.92) < 0.5,
    "Task 6 complete. 14 days of 31 exceed the limit, and the mean is 60.92. "
    "Now go back and read your answer to Task 3 again.",
    "Not right yet. Expected 14 days above the limit and a mean near 60.92.",
)

### Go Back And Look At Task 3

You have an answer. Fourteen days of January exceeded the European daily limit,
and the month averaged 60.92 micrograms per cubic metre. Every reading that
went into it survived a range check and a location check, and no value in the
series is missing.

Now recall the number you were told to set aside. Station 78456 reported a
median PM10 of 1.08 while the other seven stations reported between 8.50 and
60.63.

## Task 7: Measure What That Station Cost You

Station 78456 is not measuring Skopje's air. A working outdoor sensor in that
city in January does not report a median of 1.08 when every neighbour reports
eight to fifty-six times more. It is indoors, or filtered, or broken.
Which of the three does not matter, because none of them is the quantity you
are reporting.

Note what it survived. It has no missing values, it reported on all 31 days and
no station beats that, every reading is positive, and none is anywhere near the
sensor's ceiling. It fails no range check, no null check, and
no coordinate check. It is the tidiest column in the table.

Recompute the whole city series without it. Store the new count in
`honest_days_over_limit` and the new hourly series in `honest_hourly`, then
plot the two daily series together.

In [ ]:
# YOUR CODE HERE
# Drop station 78456 from `hourly`, recompute the city series, and store
# `honest_hourly` and `honest_days_over_limit`. Plot both daily series.

In [ ]:
check(
    "honest_days_over_limit" in dir()
    and honest_days_over_limit == 16
    and "honest_hourly" in dir()
    and abs(honest_hourly.mean() - 71.40) < 0.5,
    "Task 7 complete. 16 days exceeded the limit, not 14, and the mean is 71.40.",
    "Not right yet. Expected 16 days above the limit and a mean near 71.40.",
)

### What Just Happened

One sensor in seven, contributing a seventh of most hourly averages, pulled the
month's mean down from 71.40 to 60.92 and hid two days of exceedance. Reported
as it stood, the answer understates Skopje's January by just under fifteen
percent and undercounts the days on which a legal limit was breached.

Nothing in the modelling was wrong, because there was no modelling. The whole
error was committed during data preparation, and it was committed in Task 3, at
the moment you saw the median of 1.08 and moved on.

The general shape of this is worth carrying. Task 5's defect was caught by a
rule, because a station in Belgium violates a fact about geography that can be
written down. Task 7's defect violates no rule at all. It was found by
comparing a sensor against its neighbours, which is the only check that would
have found it, and that check exists only if somebody thinks to make it.

Averaging is what made it dangerous. A mean across seven stations gives a
broken one a full seventh of the answer and reports no sign of distress. Look
at the parts before you combine them, because the combination is where evidence
goes to disappear.

## Task 8: Save The Dataset

The point of a day spent preparing data is the dataset at the end of it. Write
`honest_hourly` out as a two-column CSV of `timestamp` and `pm10`, into
`DATA_DIR / "city_hourly_pm10.csv"`.

It is an hourly, regularly spaced, single-valued series covering January 2024,
which is the shape any time series method expects and none of the eight
original files had.

In [ ]:
output = honest_hourly.rename("pm10").to_frame()
output.index.name = "timestamp"
output.to_csv(DATA_DIR / "city_hourly_pm10.csv")

print("Rows written:", len(output))
print("Missing values:", int(output["pm10"].isna().sum()))
print("Spacing is regular:", output.index.to_series().diff().dropna().nunique() == 1)
output.head()

## Extensions

Work on these in any order if you have time left.

1. The mean across stations gives every station an equal vote in every hour,
   but station 78188 reported on only 24 days. Work out which hours are
   averaged over fewer stations than others, and decide whether that changes
   the answer to Task 7.
2. Use the median across stations instead of the mean, without removing station
   78456. Does the median resist one broken sensor in seven well enough to
   recover the right answer on its own?
3. Write a function that flags a station whose readings correlate poorly with
   the rest of the network over the same hours. Check whether it would have
   found station 78456 without anybody looking at the table.
4. PM2.5 was carried through the whole laboratory exercise and never used.
   Compute the ratio of PM2.5 to PM10 per station and see whether station 78456
   stands out there too.
5. Resample to a daily mean directly from the raw readings, without the
   intermediate hourly step, and explain why the two answers differ.

## What To Take Away

- Look at every source separately before you combine them. An average across
  sensors is exactly the operation that conceals a broken sensor.
- A column with no missing values, no outliers, and complete coverage can still
  be entirely wrong. Tidiness is not correctness.
- Parse timestamps to a single explicit timezone. A mixed column parses into
  `object` dtype without raising, and fails later somewhere unrelated.
- A repeated value at the top of an instrument's range is the instrument
  refusing to answer, not a measurement.
- Treat metadata as data and check it. Coordinates, identifiers, and units all
  arrive wrong sometimes.
- Say which way an unavoidable error runs. Both available treatments of the
  saturated readings bias the month downward, and the reader needs to know
  that.
- Write the decisions down. Every repair in this laboratory exercise changed
  the published answer, and none of them left a trace in the output.